# Thème Numéro 1 - La Perception de Soi

## Breakdown:
- Q1 Les personnes qui s'évaluent mieux obtiennent-elles plus de matchs?
- Q2 Vaut-il mieux être confiant ou réaliste?
- Q3 Le succès lors du speed-dating (nombre de matchs obtenus) influence-t-il la perception de soi après l'événement?

## Question 2 - Vaut-il mieux être confiant ou réaliste?

- **H0** : il n'y a pas de différence de nombre moyen de matchs entre les 3 groupes
- **H1** : il existe au moins une différence significative entre les groupes

Seuil de significativité : α = 0.05

## 0. Chargement des données

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

In [2]:
df = pd.read_csv("Speed+Dating+Data.csv",encoding="MacRoman")
# display(df.head())
df.info()
#df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 12.5+ MB


In [3]:
## créer une copie du dataframe pour uniquement avoir une ligne par personne
df_1_2 = df.copy()
df_1_2.head()

,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


In [4]:
## garder uniquement les colonnes qu'on veut
cols_to_keep = [ "iid","match", 
    "attr3_1", "sinc3_1", "intel3_1", "fun3_1", "amb3_1", 
    "attr_o", "sinc_o", "intel_o", "fun_o", "amb_o"
]

df_1_2 = df_1_2[cols_to_keep]
df_1_2.head()

,iid,match,attr3_1,sinc3_1,intel3_1,fun3_1,amb3_1,attr_o,sinc_o,intel_o,fun_o,amb_o
0,1,0,6.0,8.0,8.0,8.0,7.0,6.0,8.0,8.0,8.0,8.0
1,1,0,6.0,8.0,8.0,8.0,7.0,7.0,8.0,10.0,7.0,7.0
2,1,1,6.0,8.0,8.0,8.0,7.0,10.0,10.0,10.0,10.0,10.0
3,1,1,6.0,8.0,8.0,8.0,7.0,7.0,8.0,9.0,8.0,9.0
4,1,1,6.0,8.0,8.0,8.0,7.0,8.0,7.0,9.0,6.0,9.0


In [5]:
# df_self = df_self.dropna()
df_1_2.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   iid       8378 non-null   int64  
 1   match     8378 non-null   int64  
 2   attr3_1   8273 non-null   float64
 3   sinc3_1   8273 non-null   float64
 4   intel3_1  8273 non-null   float64
 5   fun3_1    8273 non-null   float64
 6   amb3_1    8273 non-null   float64
 7   attr_o    8166 non-null   float64
 8   sinc_o    8091 non-null   float64
 9   intel_o   8072 non-null   float64
 10  fun_o     8018 non-null   float64
 11  amb_o     7656 non-null   float64
dtypes: float64(10), int64(2)
memory usage: 785.6 KB


Beaucoup de valeurs nulles, notamment pour les notes reçues.

Colonnes auto-évaluation → 105 null (~1.2%)
Colonnes évaluations reçues → entre 287 et 722 null (jusqu'à ~8.6%).

## 1. Gestion des valeurs manquantes

Imputation par la médiane par iid :
- D'abord on essaye de remplir avec les autres valeurs du même individu (médiane par iid).
- Ensuite, si un individu n'a aucune valeur du tout, on supprime cet iid, car il ne peut pas contribuer à l'analyse.

**Ce choix préserve la variabilité individuelle, cohérent avec une analyse centrée sur la personne.**

In [6]:
# 1. IMPUTATION
cols_self = ['attr3_1', 'sinc3_1', 'intel3_1', 'fun3_1', 'amb3_1']
cols_received = ['attr_o', 'sinc_o', 'intel_o', 'fun_o', 'amb_o']

# Imputation par la médiane de chaque iid
for col in cols_self + cols_received:
    df_1_2[col] = df_1_2.groupby('iid')[col].transform(
        lambda x: x.fillna(x.median())
    )

# S'il reste des NaN, drop iid
df_1_2 = df_1_2.dropna(subset=cols_self + cols_received)

df_1_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8273 entries, 0 to 8377
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   iid       8273 non-null   int64  
 1   match     8273 non-null   int64  
 2   attr3_1   8273 non-null   float64
 3   sinc3_1   8273 non-null   float64
 4   intel3_1  8273 non-null   float64
 5   fun3_1    8273 non-null   float64
 6   amb3_1    8273 non-null   float64
 7   attr_o    8273 non-null   float64
 8   sinc_o    8273 non-null   float64
 9   intel_o   8273 non-null   float64
 10  fun_o     8273 non-null   float64
 11  amb_o     8273 non-null   float64
dtypes: float64(10), int64(2)
memory usage: 840.2 KB


## 2. Variables par personne
Pour chaque individu, on calcule :
- **self_perception** : moyenne globale des auto-évaluations (attr, sinc, intel, fun, amb)
- **others_perception** : moyenne globale des évaluations reçues par les partenaires
- **nb_matchs** : nombre de rencontres où `match == 1`

In [7]:
# 2. VARIABLES PAR PERSONNE
per_person = df_1_2.groupby('iid').agg(
    self_perception = (cols_self[0], 'mean'),
    others_perception = (cols_received[0], 'mean'),
    nb_matchs = ('match', 'sum')
).reset_index()

# Moyenne sur toutes les dimensions self & received
## df_1_2.groupby('iid')[cols_self].mean(): calcule la moyenne de chaque colonne séparément pour chaque iid
## .mean(axis=1): pour chaque personne (axis=1 = sur les colonnes), fait la moyenne des 5 dimensions
## .values: convertit Series en array pour l'assigner proprement à per_person
per_person['self_perception'] = df_1_2.groupby('iid')[cols_self].mean().mean(axis=1).values
per_person['others_perception'] = df_1_2.groupby('iid')[cols_received].mean().mean(axis=1).values

per_person.head()

,iid,self_perception,others_perception,nb_matchs
0,1,7.4,7.46,4
1,2,6.6,7.54,2
2,3,8.4,6.84,0
3,4,7.8,7.40,2
4,5,6.6,7.12,2


## 3. Score de biais

Le biais est défini comme la différence entre l'auto-perception et l'évaluation reçue :

``` python
bias = self_perception - others_perception
```
- **bias > 0** : la personne se surestime par rapport à comment elle est perçue
- **bias ≈ 0** : la personne est réaliste
- **bias < 0** : la personne se sous-estime

In [8]:
# 3. SCORE DE BIAIS
per_person['bias'] = per_person['self_perception'] - per_person['others_perception']

per_person.head()

,iid,self_perception,others_perception,nb_matchs,bias
0,1,7.4,7.46,4,-0.06
1,2,6.6,7.54,2,-0.94
2,3,8.4,6.84,0,1.56
3,4,7.8,7.40,2,0.40
4,5,6.6,7.12,2,-0.52


## 4. Création des groupes

On segmente les individus en 3 groupes selon leur score de biais, en utilisant l'écart-type comme seuil.

- **sous-estimation** : biais < -std/2
- **réaliste** : biais entre -std/2 et std/2
- **surestimation** : biais > std/2

Ce choix ancre les groupes autour de 0, ce qui correspond à la définition conceptuelle du biais :
un individu "réaliste" est quelqu'un dont l'auto-perception est proche de l'évaluation reçue,
peu importe la distribution globale.

In [9]:
# per_person = per_person.drop(columns=['groupe'])

In [10]:
# 4. GROUPES
std = per_person['bias'].std()

per_person['group'] = pd.cut(
    per_person['bias'],
    bins=[-np.inf, -std/2, std/2, np.inf],
    labels=['sous-estimation', 'réaliste', 'surestimation']
)

per_person.head()

,iid,self_perception,others_perception,nb_matchs,bias,group
0,1,7.4,7.46,4,-0.06,réaliste
1,2,6.6,7.54,2,-0.94,sous-estimation
2,3,8.4,6.84,0,1.56,surestimation
3,4,7.8,7.40,2,0.40,réaliste
4,5,6.6,7.12,2,-0.52,réaliste


## 5. Analyse descriptive

Comparaison du nombre moyen de matchs entre les 3 groupes.

In [11]:
# 5. ANALYSE DESCRIPTIVE
print("Infos analytiques des groupes:")
print(per_person.groupby('group')['nb_matchs'].describe())
print("\nMoyenne de matchs par groupe:")
print(per_person.groupby('group')['nb_matchs'].mean())

most_match_group = per_person.groupby('group')['nb_matchs'].mean().idxmax()
print(f"\n→ Le groupe avec le plus de matchs est : {most_match_group}")

Infos analytiques des groupes:
                 count      mean       std  min  25%  50%  75%   max
group                                                               
sous-estimation   35.0  2.828571  2.107211  0.0  1.0  2.0  4.0   8.0
réaliste         153.0  2.751634  2.560533  0.0  1.0  2.0  4.0  14.0
surestimation    354.0  2.347458  2.154291  0.0  1.0  2.0  3.0  11.0

Moyenne de matchs par groupe:
group
sous-estimation    2.828571
réaliste           2.751634
surestimation      2.347458
Name: nb_matchs, dtype: float64

→ Le groupe avec le plus de matchs est : sous-estimation


C:\Users\PC\AppData\Local\Temp\ipykernel_37804\4134625736.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(per_person.groupby('group')['nb_matchs'].describe())
C:\Users\PC\AppData\Local\Temp\ipykernel_37804\4134625736.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(per_person.groupby('group')['nb_matchs'].mean())
C:\Users\PC\AppData\Local\Temp\ipykernel_37804\4134625736.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and sile

## 6. Test ANOVA

- **H0** : il n'y a pas de différence de nombre moyen de matchs entre les 3 groupes
- **H1** : il existe au moins une différence significative entre les groupes

- **Variable dépendante** : nombre de matchs
- **Variable indépendante** : groupe de biais (surestimation / réaliste / sous-estimation)

Seuil de significativité : α = 0.05

In [12]:
# 6. ANOVA
groups = [group['nb_matchs'].values
           for _, group in per_person.groupby('group')]

f_stat, p_value = stats.f_oneway(*groups)
print(f"\nResultats ANOVA:\nF = {f_stat:.3f} \np = {p_value:.4f}")

if p_value < 0.05:
    print("H0 rejetée: différence significative entre les groupes.")
else:
    print("H0 non-rejetée: pas de différence significative entre les groupes.")


Resultats ANOVA:
F = 2.097 
p = 0.1239
H0 non-rejetée: pas de différence significative entre les groupes.


C:\Users\PC\AppData\Local\Temp\ipykernel_37804\3832581636.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, group in per_person.groupby('group')]


## 7. Visualisation

Le graphique ci-dessous illustre la moyenne de matchs pour chaque groupe de biais.
On observe que les trois barres sont proches, ce qui est cohérent avec le résultat de l'ANOVA (p = 0.124) — aucun groupe ne se démarque clairement des autres.

In [13]:

group_avgs = per_person.groupby('group')['nb_matchs'].mean().reset_index()

fig = px.bar(
    group_avgs,
    x='group',
    y='nb_matchs',
    color='group',
    text='nb_matchs',                         
    title="Nombre moyen de matchs par groupe de biais",
    labels={'nb_matchs': 'Moyenne de matchs', 'group': 'Groupe'},
    category_orders={'group': ['sous-estimation', 'réaliste', 'surestimation']}  
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_37804\1094092696.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_avgs = per_person.groupby('group')['nb_matchs'].mean().reset_index()


c:\Users\PC\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## Conclusion

L'ANOVA ne révèle pas de différence significative du nombre de matchs entre les trois groupes (F = 2.097, p = 0.124). On ne peut donc pas rejeter H0.

**On ne peut pas affirmer statistiquement qu'il vaut mieux être confiant, réaliste ou modeste pour obtenir plus de matchs.**

Comme le confirme visuellement le graphique, les trois groupes obtiennent des moyennes de matchs très proches — la perception de soi ne semble pas être un facteur déterminant dans le succès en speed dating.

### Limites à considérer
- Le speed dating est un contexte artificiel qui ne reflète pas forcément les dynamiques relationnelles réelles
- La taille des groupes et la distribution du biais peuvent affecter la puissance du test

### Ce que ça nous dit quand même
Bien que non significatif, le résultat suggère que **la perception de soi n'est pas le facteur déterminant** dans le succès en speed dating — ce qui est en soi une conclusion intéressante.